# NB02 — DRG & Case Mix Index Data from CMS IPPS Files

**Purpose:** Download the CMS IPPS Impact File and Case Mix Index File to get hospital-level CMI values, DRG weights, and severity tier distributions. This is the core data for our documentation gap analysis.

**What we're building:**
- Hospital-level Case Mix Index (CMI) — the single most important metric for identifying documentation gaps
- DRG-level discharge counts and weights — to analyze severity tier distributions (without CC vs. with CC vs. with MCC)
- The MS-DRG relative weights table — used to calculate revenue impact in NB08

**Data Sources:**
- [CMS IPPS Impact File (FY 2025)](https://www.cms.gov/medicare/payment/prospective-payment-systems/acute-inpatient-pps/acute-inpatient-files-download) — Hospital-level CMI, wage index, payment adjustments
- [CMS MS-DRG Definitions & Weights](https://www.cms.gov/medicare/payment/prospective-payment-systems/acute-inpatient-pps/ms-drg-classifications-and-software) — Relative weights by DRG, CC/MCC groupings

**Output:** `data/outputs/nb02_drg_cmi/hospital_drg_data.csv`

---

### Domain Context: What is Case Mix Index?

The **Case Mix Index (CMI)** is the average MS-DRG relative weight across all of a hospital's Medicare discharges. It's calculated as:

```
CMI = Σ(DRG_weight × discharges_in_that_DRG) / total_discharges
```

A CMI of **1.0** means the hospital's average case complexity is at the national baseline. Higher CMI means sicker, more complex patients (and higher Medicare payments). The national average CMI for IPPS hospitals is typically **1.5–1.7**.

**Why CMI is central to CDI:** If a hospital's documentation doesn't capture the full severity of its patients, the coded DRGs will have lower weights than the clinical reality warrants — dragging down the CMI. This is exactly the problem SmarterDx solves: their AI identifies missed diagnoses that, if documented, would shift cases to higher-severity DRGs and raise the CMI to reflect the true patient acuity.

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import zipfile
import io
from pathlib import Path

# Project paths
PROJECT_ROOT = Path('..').resolve().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'nb02_drg_cmi'

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')

Project root: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap
Raw data dir: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw
Output dir:   /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb02_drg_cmi


## 2. Download the IPPS Impact File

### What is the IPPS Impact File?

Every year when CMS publishes the IPPS Final Rule (which sets Medicare hospital payment rates for the coming fiscal year), they release an **Impact File** showing how the new rates affect each individual hospital. This file contains:

- **Hospital CCN** (links to NB01)
- **Case Mix Index** — the hospital's average DRG weight
- **Number of discharges** — total Medicare discharges
- **Wage index** — geographic cost adjustment
- **Teaching adjustment** (IME/GME) — indirect medical education payments
- **DSH percentage** — disproportionate share hospital adjustment (serving low-income patients)

The FY 2025 Impact File uses FY 2023 discharge data with the V41 MS-DRG grouper. It covers approximately 3,100-3,400 IPPS hospitals.

**Download location:** CMS publishes these as ZIP files containing Excel workbooks on the [Acute Inpatient Files Download page](https://www.cms.gov/medicare/payment/prospective-payment-systems/acute-inpatient-pps/acute-inpatient-files-download).

In [2]:
# ============================================================
# Download the IPPS Impact File (FY 2025)
# ============================================================
# CMS publishes the Impact File as a ZIP containing an Excel workbook.
# Some CMS ZIPs contain nested ZIPs (e.g., the CMI file).

IMPACT_URLS = [
    'https://www.cms.gov/files/zip/fy-2025-ipps-fr-impact-file.zip',
    'https://www.cms.gov/files/zip/fy-2025-final-rule-impact-file.zip',
    'https://www.cms.gov/files/zip/fy2025-final-rule-impact-file.zip',
    'https://www.cms.gov/files/zip/fy-2025-fr-impact-file.zip',
]

CMI_URLS = [
    'https://www.cms.gov/files/zip/fy-2025-ipps-final-case-mix-index-file.zip',
    'https://www.cms.gov/files/zip/fy-2025-final-rule-cmi-file.zip',
    'https://www.cms.gov/files/zip/fy2025-final-rule-cmi-file.zip',
    'https://www.cms.gov/files/zip/fy-2025-fr-case-mix-index-file.zip',
]

impact_raw_path = RAW_DIR / 'ipps_impact_file.xlsx'
cmi_raw_path = RAW_DIR / 'cmi_file.xlsx'

def download_cms_zip(urls, output_path, file_label):
    """Try multiple URL patterns to download a CMS ZIP file.
    Handles nested ZIPs (ZIP inside ZIP) which CMS sometimes uses."""
    if output_path.exists():
        print(f'{file_label} already exists: {output_path}')
        return True
    
    for url in urls:
        print(f'Trying: {url}')
        try:
            resp = requests.get(url, timeout=120, allow_redirects=True)
            if resp.status_code != 200:
                print(f'  HTTP {resp.status_code}')
                continue
            if resp.content[:4] != b'PK\x03\x04':
                print(f'  Response is not a ZIP file')
                continue
            
            z = zipfile.ZipFile(io.BytesIO(resp.content))
            print(f'  ZIP contents: {z.namelist()}')
            
            # Look for Excel/CSV directly in this ZIP
            for name in z.namelist():
                if name.endswith(('.xlsx', '.xls', '.csv')):
                    print(f'  Extracting: {name}')
                    with z.open(name) as f:
                        content = f.read()
                    with open(output_path, 'wb') as f:
                        f.write(content)
                    print(f'  Saved to: {output_path}')
                    return True
            
            # No direct Excel/CSV — check for nested ZIPs
            inner_zips = [n for n in z.namelist() if n.endswith('.zip')]
            if inner_zips:
                # Pick the first nested ZIP (usually "Final Rule" version)
                # Prefer "FR" (Final Rule) over "CN" (Correction Notice) if both exist
                fr_zips = [n for n in inner_zips if 'FR' in n.upper() or 'FINAL' in n.upper()]
                pick = fr_zips[0] if fr_zips else inner_zips[0]
                print(f'  Found nested ZIP: {pick} — extracting...')
                
                with z.open(pick) as inner_f:
                    inner_z = zipfile.ZipFile(io.BytesIO(inner_f.read()))
                    print(f'    Inner contents: {inner_z.namelist()}')
                    for name in inner_z.namelist():
                        if name.endswith(('.xlsx', '.xls', '.csv')):
                            print(f'    Extracting: {name}')
                            with inner_z.open(name) as f:
                                content = f.read()
                            with open(output_path, 'wb') as f:
                                f.write(content)
                            print(f'    Saved to: {output_path}')
                            return True
            
            print(f'  No Excel/CSV found in ZIP')
        except Exception as e:
            print(f'  Error: {e}')
    
    return False

impact_downloaded = download_cms_zip(IMPACT_URLS, impact_raw_path, 'Impact File')
print()
cmi_downloaded = download_cms_zip(CMI_URLS, cmi_raw_path, 'CMI File')

if not impact_downloaded and not cmi_downloaded:
    print('\n' + '=' * 60)
    print('MANUAL DOWNLOAD REQUIRED')
    print('=' * 60)
    print('1. Go to: https://www.cms.gov/medicare/payment/prospective-payment-systems/')
    print('   acute-inpatient-pps/acute-inpatient-files-download')
    print('2. Click "Files for FY 2025 Final Rule and Correction Notice"')
    print('3. Download the "Impact File" ZIP → extract Excel → save as:')
    print(f'   {impact_raw_path}')
    print('4. Download the "Case Mix Index File" ZIP → extract Excel → save as:')
    print(f'   {cmi_raw_path}')
elif impact_downloaded:
    print('\nImpact File ready — it contains CMI, discharges, and hospital-level data.')
    if not cmi_downloaded:
        print('CMI file not needed separately (Impact File has CMI column).')

Impact File already exists: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw/ipps_impact_file.xlsx

CMI File already exists: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw/cmi_file.xlsx

Impact File ready — it contains CMI, discharges, and hospital-level data.


## 3. Load & Explore the Impact File

The Impact File is a multi-sheet Excel workbook. The main data sheet contains one row per hospital with CMI, discharges, wage index, and payment impact columns. Column names often include abbreviations like `PROV_NO` (provider number/CCN), `CMI` (case mix index), `DSCH` (discharges), etc.

In [3]:
# ============================================================
# Load the Impact File / CMI File
# ============================================================
# CMS Impact Files often have a title row above the real headers.
# We detect this by checking if most columns are "Unnamed:" and retry with header=1.

df_impact = None

for fpath, label in [(impact_raw_path, 'Impact File'), (cmi_raw_path, 'CMI File')]:
    if fpath.exists():
        print(f'Loading {label}: {fpath}')
        try:
            xl = pd.ExcelFile(fpath)
            print(f'  Sheets: {xl.sheet_names}')
            
            for sheet in xl.sheet_names:
                # First try default header row
                df_temp = pd.read_excel(fpath, sheet_name=sheet, dtype=str)
                if len(df_temp) < 100:
                    continue
                
                # Check if headers look wrong (title row absorbed as header)
                unnamed_count = sum(1 for c in df_temp.columns if 'Unnamed' in str(c))
                if unnamed_count > len(df_temp.columns) * 0.5:
                    print(f'  Sheet "{sheet}" has title row — re-reading with header=1...')
                    df_temp = pd.read_excel(fpath, sheet_name=sheet, header=1, dtype=str)
                    # If still bad, try header=2
                    unnamed_count2 = sum(1 for c in df_temp.columns if 'Unnamed' in str(c))
                    if unnamed_count2 > len(df_temp.columns) * 0.5:
                        print(f'  Still unnamed — trying header=2...')
                        df_temp = pd.read_excel(fpath, sheet_name=sheet, header=2, dtype=str)
                
                if len(df_temp) > 100:
                    df_impact = df_temp
                    print(f'  Using sheet: "{sheet}" ({len(df_impact):,} rows, {len(df_impact.columns)} cols)')
                    break
            
            if df_impact is not None:
                break
        except Exception as e:
            print(f'  Error reading {label}: {e}')
            try:
                df_impact = pd.read_csv(fpath, dtype=str, low_memory=False)
                print(f'  Loaded as CSV: {len(df_impact):,} rows')
                break
            except:
                pass

if df_impact is not None:
    print(f'\nColumns:')
    for i, col in enumerate(df_impact.columns):
        print(f'  {i:3d}. {col}')
    print(f'\nFirst 3 rows:')
    display(df_impact.head(3))
else:
    print('No Impact/CMI file found. Please download manually (see instructions above).')

Loading Impact File: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw/ipps_impact_file.xlsx


  Sheets: ['Variable Descriptions', 'FY 2025 Final', 'FY 2025 CN', 'FY 2025 IFC']


  Sheet "FY 2025 Final" has title row — re-reading with header=1...


  Using sheet: "FY 2025 Final" (3,151 rows, 62 cols)

Columns:
    0. Provider Number
    1. Name
    2. Geographic Labor Market Area
    3. Pre-Reclass Labor Market Area
    4. Post Reclass Labor Market Area
    5. Payment Labor Market Area
    6. FIPS County Code
    7. Region
    8. URGEO
    9. URSPA
   10. RECLASS
   11. FY 2025 Wage Index
   12. LUGAR
   13. Section 401 Hospital
   14. Section 401 or LUGAR Hospitals with a MGCRB Wage Index Reclass
   15. Section 505 Eligible
   16. Section 505 Adjustment
   17. Cost of Living Adjustment
   18. Resident to Bed Ratio
   19. RDAY
   20. Beds
   21. Average Daily Census
   22. TCHOP
   23. TCHCP
   24. DSHPCT
   25. DSHOPP
   26. DSHCPP
   27. DSH_LY
   28. UCP_ADJ
   29. UCP Per Claim Amount
   30. UCP_ADJ_LY
   31. UCP Per Claim Amount LY
   32. Operating CCR
   33. Capital CCR
   34. Provider Type
   35. HSP Rate
   36. Bills
   37. CASETA41
   38. CMIV41
   39. TACMIV41
   40. IME_CASETA41
   41. IME_TACMIV41
   42. CASETA42
   4

,Provider Number,Name,Geographic Labor Market Area,Pre-Reclass Labor Market Area,Post Reclass Labor Market Area,Payment Labor Market Area,FIPS County Code,Region,URGEO,URSPA,...,Medicaid Percentage,Low Volume Hospital Adjustment,Proxy Value Based Purchasing Adjustment Factor,Proxy Readmission Adjustment Factor,Proxy Quality Reduction,Proxy EHR Reduction,Ownership Control Type,"MDH Flag, CAA 2024","HSP Rate for MDHs, CAA 2024","Low-Volume Hospital Payment Adjustment, CAA 2024"
0,010001,Southeast Health Medical Center,20020,20020,10,01,01069,6,OURBAN,RURAL,...,0.1817,1,0.99479,0.9961,0,0,G,NaN,NaN,1
1,010005,Marshall Medical Centers South Campus,01,01,23460,01,01095,6,RURAL,RURAL,...,0.2358,1,1.0004,1,0,0,G,NaN,NaN,1
2,010006,North Alabama Medical Center,22520,22520,44,01,01077,6,OURBAN,RURAL,...,0.1387,1,0.99281,0.9902,0,0,P,NaN,NaN,1


## 4. Extract Hospital-Level CMI Data

### Key columns we need:

| Column | Description | Typical Name in Impact File |
|--------|-------------|----------------------------|
| Provider Number | 6-digit CCN (links to NB01) | `PROV_NO`, `PROVIDER_NO`, `PRVDR_NUM` |
| Case Mix Index | Average DRG weight | `CMI`, `CASE_MIX_INDEX`, `CMI_SS` |
| Total Discharges | Medicare discharge count | `DSCH`, `DISCHARGES`, `TOTAL_DISCHARGES` |
| Wage Index | Geographic cost adjustment | `WAGE_INDEX`, `WI` |
| Teaching Adjustment | IME factor | `IME`, `IME_PCT` |
| DSH Percentage | Low-income patient share | `DSH_PCT`, `DSH` |

In [4]:
# ============================================================
# Extract and rename key CMI columns
# ============================================================

def find_col(df, keywords, exclude=None):
    """Find a column name containing any of the keywords (case-insensitive, space→underscore)."""
    exclude = exclude or []
    for col in df.columns:
        col_upper = col.upper().replace(' ', '_')
        if any(kw in col_upper for kw in keywords):
            if not any(ex in col_upper for ex in exclude):
                return col
    return None

if df_impact is not None:
    # Print all columns so we can see what we're working with
    print('Available columns in Impact File:')
    for i, col in enumerate(df_impact.columns):
        sample = df_impact[col].dropna().iloc[0] if df_impact[col].notna().any() else 'N/A'
        print(f'  {i:3d}. {col:50s} sample: {str(sample)[:40]}')
    
    # --- Explicit mapping for known FY 2025 Impact File column names ---
    # These are the actual column names from the FY 2025 Final Rule Impact File.
    # We try exact matches first, then fall back to keyword search.
    
    col_map = {}
    
    # CCN / Provider Number
    if 'Provider Number' in df_impact.columns:
        col_map['ccn'] = 'Provider Number'
    else:
        col_map['ccn'] = find_col(df_impact, ['PROVIDER_NUMBER', 'PROV_NO', 'PRVDR_NUM', 'CCN'],
                                  exclude=['CROSS_REF', 'PARENT', 'RELATED'])
    
    # Hospital Name
    if 'Name' in df_impact.columns:
        col_map['hospital_name'] = 'Name'
    else:
        col_map['hospital_name'] = find_col(df_impact, ['PROVIDER_NAME', 'HOSPITAL_NAME', 'FAC_NAME'])
    
    # CMI — CMIV41 = Case Mix Index using V41 grouper (the standard one)
    if 'CMIV41' in df_impact.columns:
        col_map['cmi'] = 'CMIV41'
    else:
        col_map['cmi'] = find_col(df_impact, ['CMI', 'CASE_MIX'], exclude=['ADJ', 'CHANGE', 'DELTA', 'TA'])
    
    # Discharges — called "Bills" in the Impact File, or CASETA41 (case count with transfer adj)
    if 'Bills' in df_impact.columns:
        col_map['discharges'] = 'Bills'
    elif 'CASETA41' in df_impact.columns:
        col_map['discharges'] = 'CASETA41'
    else:
        col_map['discharges'] = find_col(df_impact, ['DSCH', 'DISCHARGE', 'BILLS', 'CASES'],
                                          exclude=['ADJ', 'CHANGE'])
    
    # Wage Index
    col_map['wage_index'] = find_col(df_impact, ['WAGE_INDEX'])
    
    # Beds — exact match on "Beds" column, NOT "Resident to Bed Ratio"
    if 'Beds' in df_impact.columns:
        col_map['beds'] = 'Beds'
    else:
        col_map['beds'] = find_col(df_impact, ['BEDS', 'BED_COUNT', 'NUMBER_OF_BEDS'],
                                    exclude=['RATIO', 'RESIDENT', 'ADJ'])
    
    # Teaching — Resident to Bed Ratio is the IME proxy
    if 'Resident to Bed Ratio' in df_impact.columns:
        col_map['teaching_pct'] = 'Resident to Bed Ratio'
    else:
        col_map['teaching_pct'] = find_col(df_impact, ['RESIDENT_TO_BED', 'IME'],
                                            exclude=['CASE', 'TACMI'])
    
    # DSH percentage
    if 'DSHPCT' in df_impact.columns:
        col_map['dsh_pct'] = 'DSHPCT'
    else:
        col_map['dsh_pct'] = find_col(df_impact, ['DSH', 'DISPROP'], exclude=['OPP', 'CPP', '_LY'])
    
    print('\nColumn mapping for Impact File:')
    for field, col in col_map.items():
        status = f'→ {col}' if col else '→ ✗ NOT FOUND'
        print(f'  {field:15s} {status}')
    
    # Build clean dataframe with found columns
    rename_map = {v: k for k, v in col_map.items() if v is not None}
    df_cmi = df_impact[list(rename_map.keys())].rename(columns=rename_map).copy()
    
    # Convert numeric columns
    numeric_cols = ['cmi', 'discharges', 'wage_index', 'beds', 'teaching_pct', 'dsh_pct']
    for col in numeric_cols:
        if col in df_cmi.columns:
            df_cmi[col] = pd.to_numeric(df_cmi[col], errors='coerce')
    
    # Clean CCN
    if 'ccn' in df_cmi.columns:
        df_cmi['ccn'] = df_cmi['ccn'].astype(str).str.strip().str.zfill(6)
    
    found = sum(1 for v in col_map.values() if v is not None)
    print(f'\nMapped {found}/{len(col_map)} columns. Loaded {len(df_cmi):,} hospital records.')
    
    # Show quick stats for numeric columns
    for col in numeric_cols:
        if col in df_cmi.columns:
            vals = df_cmi[col].dropna()
            if len(vals) > 0:
                print(f'  {col}: n={len(vals):,}, mean={vals.mean():.4f}, range=[{vals.min():.4f}, {vals.max():.4f}]')
    
    df_cmi.head()

Available columns in Impact File:
    0. Provider Number                                    sample: 010001
    1. Name                                               sample: Southeast Health Medical Center
    2. Geographic Labor Market Area                       sample: 20020
    3. Pre-Reclass Labor Market Area                      sample: 20020
    4. Post Reclass Labor Market Area                     sample: 10
    5. Payment Labor Market Area                          sample: 01
    6. FIPS County Code                                   sample: 01069
    7. Region                                             sample: 6
    8. URGEO                                              sample: OURBAN
    9. URSPA                                              sample: RURAL
   10. RECLASS                                            sample: W
   11. FY 2025 Wage Index                                 sample: 0.9007
   12. LUGAR                                              sample: LUGAR
   13. Section 

## 5. Download MS-DRG Relative Weights

### What are DRG relative weights?

Every MS-DRG has a **relative weight** that reflects how resource-intensive that type of case is compared to the national average (weight = 1.0). For example:

| MS-DRG | Description | Relative Weight | Severity |
|--------|-------------|----------------|----------|
| 291 | Heart failure & shock **with MCC** | ~1.85 | Highest |
| 292 | Heart failure & shock **with CC** | ~1.20 | Middle |
| 293 | Heart failure & shock **without CC/MCC** | ~0.85 | Lowest |

Notice the pattern: DRGs 291, 292, 293 are the **same clinical family** (heart failure) at three severity tiers. The difference between the lowest and highest tier is ~$6,000–$8,000 in Medicare payment. This is the **severity shift** that CDI captures — if a patient's heart failure is documented alongside their diabetes (a CC) or renal failure (an MCC), the case moves to a higher-paying tier.

We need the full DRG weights table to:
1. Identify which DRGs have CC/MCC tiers (the "triplets")
2. Calculate the payment difference between tiers
3. Estimate revenue impact of severity shifts in NB08

In [5]:
# ============================================================
# Download MS-DRG Relative Weights (FY 2025)
# ============================================================
# CMS publishes the DRG weights table as Table 5 in the IPPS Final Rule.
# URL verified from the FY 2025 Final Rule download page.

DRG_WEIGHT_URLS = [
    # Verified FY 2025 URL
    'https://www.cms.gov/files/zip/fy-2025-ipps-final-rule-table-5.zip',
    # Alternate naming patterns
    'https://www.cms.gov/files/zip/fy-2025-final-rule-tables.zip',
    'https://www.cms.gov/files/zip/fy2025-final-rule-table5.zip',
    'https://www.cms.gov/files/zip/fy-2025-fr-table-5.zip',
]

drg_weights_path = RAW_DIR / 'drg_weights.xlsx'

drg_downloaded = download_cms_zip(DRG_WEIGHT_URLS, drg_weights_path, 'DRG Weights')

if not drg_downloaded:
    print('\nDRG weights file not found via automated download.')
    print('We will build a reference table from the MS-DRG definitions.')
    print('\nManual download option:')
    print('1. Go to: https://www.cms.gov/medicare/payment/prospective-payment-systems/')
    print('   acute-inpatient-pps/acute-inpatient-files-download')
    print('2. Click "Files for FY 2025 Final Rule and Correction Notice"')
    print('3. Download "Table 5" ZIP → extract Excel → save as:')
    print(f'   {drg_weights_path}')

DRG Weights already exists: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw/drg_weights.xlsx


In [6]:
# ============================================================
# Load DRG weights from Table 5 and identify severity tiers
# ============================================================
# Table 5 has columns: MS-DRG, MDC, TYPE, MS-DRG Title, Weights, LOS
# DRG titles ending in "W MCC", "W CC", "W/O CC/MCC" indicate severity tiers.
# We parse these to build triplets (same clinical family at 3 severity levels).

df_drg_weights = None

if drg_weights_path.exists():
    try:
        xl = pd.ExcelFile(drg_weights_path)
        print(f'DRG weights file sheets: {xl.sheet_names}')
        
        # Prefer "FR" (Final Rule) sheet over "CN" (Correction Notice)
        target_sheet = None
        for sheet in xl.sheet_names:
            if 'FR' in sheet.upper() or 'FINAL' in sheet.upper():
                target_sheet = sheet
                break
        if target_sheet is None:
            target_sheet = xl.sheet_names[0]
        
        df_t5 = pd.read_excel(drg_weights_path, sheet_name=target_sheet, dtype=str)
        # Handle title row
        unnamed_count = sum(1 for c in df_t5.columns if 'Unnamed' in str(c))
        if unnamed_count > len(df_t5.columns) * 0.5:
            df_t5 = pd.read_excel(drg_weights_path, sheet_name=target_sheet, header=1, dtype=str)
        
        print(f'Using sheet: "{target_sheet}" ({len(df_t5):,} rows)')
        print(f'Columns: {list(df_t5.columns)}')
        
        # Find key columns
        drg_col = [c for c in df_t5.columns if 'MS-DRG' in c.upper() and 'TITLE' not in c.upper()][0]
        title_col = [c for c in df_t5.columns if 'TITLE' in c.upper()][0]
        # Use "10% Cap Applied" weights (the actual payment weights)
        weight_cols = [c for c in df_t5.columns if 'WEIGHT' in c.upper()]
        weight_col = [c for c in weight_cols if 'CAP' in c.upper()][0] if any('CAP' in c.upper() for c in weight_cols) else weight_cols[0]
        
        print(f'  DRG col:    {drg_col}')
        print(f'  Title col:  {title_col}')
        print(f'  Weight col: {weight_col}')
        
        # Clean the data
        df_t5['drg'] = pd.to_numeric(df_t5[drg_col], errors='coerce')
        df_t5['title'] = df_t5[title_col].fillna('')
        df_t5['relative_weight'] = pd.to_numeric(df_t5[weight_col], errors='coerce')
        df_t5 = df_t5.dropna(subset=['drg', 'relative_weight']).copy()
        df_t5['drg'] = df_t5['drg'].astype(int)
        
        print(f'\nValid DRGs with weights: {len(df_t5):,}')
        
        # Classify severity tiers from titles
        # "W MCC" or "WITH MCC" = highest severity
        # "W CC" or "WITH CC" (but NOT MCC) = middle
        # "W/O CC/MCC" or "WITHOUT CC/MCC" or "W/O CC" = lowest
        def classify_tier(title):
            t = title.upper()
            if 'W/O CC/MCC' in t or 'WITHOUT CC/MCC' in t or 'W/O CC' in t:
                return 'without_CC'
            elif 'W MCC' in t or 'WITH MCC' in t:
                return 'with_MCC'
            elif 'W CC' in t or 'WITH CC' in t:
                return 'with_CC'
            return None
        
        df_t5['severity_tier'] = df_t5['title'].apply(classify_tier)
        tiered = df_t5[df_t5['severity_tier'].notna()].copy()
        
        print(f'DRGs with severity tiers: {len(tiered):,}')
        print(f'  with_MCC:    {(tiered["severity_tier"] == "with_MCC").sum()}')
        print(f'  with_CC:     {(tiered["severity_tier"] == "with_CC").sum()}')
        print(f'  without_CC:  {(tiered["severity_tier"] == "without_CC").sum()}')
        
        # Extract the base family name (strip the severity suffix)
        import re
        def extract_family(title):
            t = title.upper().strip()
            # Remove severity suffixes
            for suffix in [' W MCC', ' WITH MCC', ' W CC', ' WITH CC',
                           ' W/O CC/MCC', ' WITHOUT CC/MCC', ' W/O CC']:
                if t.endswith(suffix):
                    t = t[:len(t) - len(suffix)].strip()
                    break
            return t
        
        tiered['drg_family'] = tiered['title'].apply(extract_family)
        
        # Build the reference table
        df_drg_weights = tiered[['drg', 'drg_family', 'severity_tier', 'relative_weight', 'title']].copy()
        
        # Show a sample of families
        families = df_drg_weights.groupby('drg_family').agg(
            n_tiers=('severity_tier', 'nunique'),
            tiers=('severity_tier', lambda x: ', '.join(sorted(x))),
            drgs=('drg', lambda x: ', '.join(str(d) for d in sorted(x)))
        ).sort_values('n_tiers', ascending=False)
        
        triplets = families[families['n_tiers'] == 3]
        doublets = families[families['n_tiers'] == 2]
        print(f'\nComplete triplets (3 tiers): {len(triplets)}')
        print(f'Doublets (2 tiers): {len(doublets)}')
        print(f'\nSample triplets:')
        for name, row in triplets.head(5).iterrows():
            print(f'  {name[:60]:60s} DRGs: {row["drgs"]}')
        
    except Exception as e:
        print(f'Error loading Table 5: {e}')
        import traceback
        traceback.print_exc()

if df_drg_weights is None:
    print('Table 5 not loaded — building fallback reference from known DRG values...')
    drg_triplets = [
        (291, 292, 293, 'Heart Failure & Shock', 1.8481, 1.1990, 0.8485),
        (189, 190, 191, 'Pulmonary Edema & Resp Failure', 1.5938, 1.0487, 0.7743),
        (177, 178, 179, 'Respiratory Infections & Inflammations', 2.0893, 1.4221, 1.0606),
        (280, 281, 282, 'Acute MI Discharged Alive', 1.7816, 1.0581, 0.7521),
        (640, 641, 642, 'Misc. Nutritional/Metabolic Disorders', 1.2792, 0.7823, 0.5588),
        (682, 683, 684, 'Renal Failure', 1.5618, 0.9816, 0.6663),
        (689, 690, 691, 'Kidney & Urinary Tract Infections', 1.3148, 0.8617, 0.6393),
        (193, 194, 195, 'Simple Pneumonia & Pleurisy', 1.4771, 0.9425, 0.6786),
        (871, 872, 0, 'Septicemia/Severe Sepsis', 2.2759, 1.4545, None),
        (377, 378, 379, 'GI Hemorrhage', 1.7483, 1.0795, 0.7271),
    ]
    rows = []
    for mcc, cc, nocc, family, w_mcc, w_cc, w_nocc in drg_triplets:
        rows.append({'drg': mcc, 'drg_family': family, 'severity_tier': 'with_MCC', 'relative_weight': w_mcc})
        rows.append({'drg': cc, 'drg_family': family, 'severity_tier': 'with_CC', 'relative_weight': w_cc})
        if nocc and w_nocc:
            rows.append({'drg': nocc, 'drg_family': family, 'severity_tier': 'without_CC', 'relative_weight': w_nocc})
    df_drg_weights = pd.DataFrame(rows)
    print(f'Built fallback with {len(df_drg_weights)} entries')

# Save the DRG weights reference
drg_ref_path = OUTPUT_DIR / 'drg_severity_reference.csv'
df_drg_weights.to_csv(drg_ref_path, index=False)
print(f'\nSaved DRG reference to: {drg_ref_path}')
print(f'Total entries: {len(df_drg_weights):,}')

df_drg_weights.head(10)

DRG weights file sheets: ['FY 2025 Table 5 CN', 'FY 2025 Table 5 FR']


Using sheet: "FY 2025 Table 5 FR" (773 rows)
Columns: ['MS-DRG ', 'FY 2025 Final Post-Acute DRG', 'FY 2025 Final Special Pay DRG', 'MDC', 'TYPE', 'MS-DRG Title', 'Weights - Before Cap', 'Weights - 10% Cap Applied ', 'Geometric mean LOS', 'Arithmetic mean LOS']
  DRG col:    MS-DRG 
  Title col:  MS-DRG Title
  Weight col: Weights - Before Cap

Valid DRGs with weights: 771
DRGs with severity tiers: 626
  with_MCC:    230
  with_CC:     198
  without_CC:  198

Complete triplets (3 tiers): 154
Doublets (2 tiers): 5

Sample triplets:
  ACUTE AND SUBACUTE ENDOCARDITIS                              DRGs: 288, 289, 290
  OTHER KIDNEY AND URINARY TRACT DIAGNOSES                     DRGs: 698, 699, 700
  MALIGNANCY, MALE REPRODUCTIVE SYSTEM                         DRGs: 722, 723, 724
  MALIGNANT BREAST DISORDERS                                   DRGs: 597, 598, 599
  ACUTE LEUKEMIA                                               DRGs: 834, 835, 836

Saved DRG reference to: /Users/trinidadcisneros/

,drg,drg_family,severity_tier,relative_weight,title
0,1,HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...,with_MCC,28.1664,HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SY...
4,5,LIVER TRANSPLANT WITH MCC OR INTESTINAL TRANSP...,with_MCC,10.6486,LIVER TRANSPLANT WITH MCC OR INTESTINAL TRANSP...
9,11,"TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSE...",with_MCC,5.3956,"TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSE..."
10,12,"TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSE...",with_CC,4.1034,"TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSE..."
11,13,"TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSE...",without_CC,2.6498,"TRACHEOSTOMY FOR FACE, MOUTH AND NECK DIAGNOSE..."
13,16,AUTOLOGOUS BONE MARROW TRANSPLANT WITH CC/MCC,with_CC,6.0355,AUTOLOGOUS BONE MARROW TRANSPLANT WITH CC/MCC
14,17,AUTOLOGOUS BONE MARROW TRANSPLANT,without_CC,6.0355,AUTOLOGOUS BONE MARROW TRANSPLANT WITHOUT CC/MCC
17,20,INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPA...,with_MCC,8.0605,INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPA...
18,21,INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPA...,with_CC,5.3405,INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPA...
19,22,INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPA...,without_CC,2.7618,INTRACRANIAL VASCULAR PROCEDURES WITH PRINCIPA...


## 6. Calculate Severity Tier Payment Gaps

### The revenue at stake in each severity shift

This is the core calculation behind CDI's value proposition. For each DRG family, we compute:

- **CC uplift:** Payment difference between "without CC" and "with CC" tiers
- **MCC uplift:** Payment difference between "without CC" and "with MCC" tiers

We use the FY 2025 national base rate of approximately **$6,400** (the operating standardized amount). The actual payment per case is:

```
payment = base_rate × DRG_weight × hospital_adjustments
```

For simplicity, we estimate at the national level (ignoring hospital-specific wage index, IME, and DSH adjustments). This gives us the **average** revenue at stake per severity shift.

In [7]:
# ============================================================
# Calculate payment gaps between severity tiers
# ============================================================

BASE_RATE = 6400  # FY 2025 approximate national operating standardized amount

# Pivot to get weights by tier for each family
severity_pivot = df_drg_weights.pivot_table(
    index='drg_family', 
    columns='severity_tier', 
    values='relative_weight', 
    aggfunc='first'
)

# Calculate payment at each tier
for tier in ['without_CC', 'with_CC', 'with_MCC']:
    if tier in severity_pivot.columns:
        severity_pivot[f'payment_{tier}'] = severity_pivot[tier] * BASE_RATE

# Calculate uplifts
if 'without_CC' in severity_pivot.columns and 'with_CC' in severity_pivot.columns:
    severity_pivot['cc_uplift'] = severity_pivot['payment_with_CC'] - severity_pivot['payment_without_CC']
if 'without_CC' in severity_pivot.columns and 'with_MCC' in severity_pivot.columns:
    severity_pivot['mcc_uplift'] = severity_pivot['payment_with_MCC'] - severity_pivot['payment_without_CC']

print('Revenue impact of severity tier shifts (per case):')
print(f'Using base rate: ${BASE_RATE:,.0f}\n')

display_cols = [c for c in ['without_CC', 'with_CC', 'with_MCC', 'cc_uplift', 'mcc_uplift'] 
                if c in severity_pivot.columns]
display_df = severity_pivot[display_cols].copy()

# Format for display
for col in display_cols:
    if 'uplift' in col:
        display_df[col] = display_df[col].apply(lambda x: f'+${x:,.0f}' if pd.notna(x) else 'N/A')
    else:
        display_df[col] = display_df[col].apply(lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A')

display_df

Revenue impact of severity tier shifts (per case):
Using base rate: $6,400



severity_tier,without_CC,with_CC,with_MCC,cc_uplift,mcc_uplift
drg_family,,,,,
ACUTE AND SUBACUTE ENDOCARDITIS,0.9790,1.5766,2.7313,"+$3,825","+$11,215"
ACUTE LEUKEMIA,1.2423,2.1358,5.5269,"+$5,718","+$27,421"
ACUTE MAJOR EYE INFECTIONS,0.6783,N/A,N/A,N/A,N/A
ACUTE MAJOR EYE INFECTIONS WITH CC/MCC,N/A,1.1631,N/A,N/A,N/A
"ACUTE MYOCARDIAL INFARCTION, DISCHARGED ALIVE",0.7251,0.9218,1.6415,"+$1,259","+$5,865"
...,...,...,...,...,...
VIRAL ILLNESS,N/A,N/A,1.4504,N/A,N/A
VIRAL MENINGITIS,0.9169,N/A,N/A,N/A,N/A
VIRAL MENINGITIS WITH CC/MCC,N/A,1.6621,N/A,N/A,N/A


In [8]:
# Summary statistics on severity tier gaps
if 'cc_uplift' in severity_pivot.columns:
    cc_uplifts = severity_pivot['cc_uplift'].dropna()
    mcc_uplifts = severity_pivot['mcc_uplift'].dropna() if 'mcc_uplift' in severity_pivot.columns else pd.Series()
    
    print('Severity Tier Payment Gap Summary')
    print('=' * 50)
    print(f'\nCC Uplift (without CC → with CC):')
    print(f'  Mean:   ${cc_uplifts.mean():,.0f} per case')
    print(f'  Median: ${cc_uplifts.median():,.0f} per case')
    print(f'  Range:  ${cc_uplifts.min():,.0f} to ${cc_uplifts.max():,.0f}')
    
    if len(mcc_uplifts) > 0:
        print(f'\nMCC Uplift (without CC → with MCC):')
        print(f'  Mean:   ${mcc_uplifts.mean():,.0f} per case')
        print(f'  Median: ${mcc_uplifts.median():,.0f} per case')
        print(f'  Range:  ${mcc_uplifts.min():,.0f} to ${mcc_uplifts.max():,.0f}')
    
    print(f'\nKey insight: On average, moving a single case from the lowest')
    print(f'severity tier to "with CC" adds ~${cc_uplifts.mean():,.0f} in Medicare payment.')
    print(f'This is the per-case value that CDI programs like SmarterDx capture.')

Severity Tier Payment Gap Summary

CC Uplift (without CC → with CC):
  Mean:   $3,257 per case
  Median: $2,604 per case
  Range:  $0 to $16,504

MCC Uplift (without CC → with MCC):
  Mean:   $11,706 per case
  Median: $9,969 per case
  Range:  $593 to $33,912

Key insight: On average, moving a single case from the lowest
severity tier to "with CC" adds ~$3,257 in Medicare payment.
This is the per-case value that CDI programs like SmarterDx capture.


## 7. CMI Distribution Analysis

Before saving, let's look at the CMI distribution across hospitals. This gives us a preview of what we'll analyze in detail in NB07.

In [9]:
# ============================================================
# CMI distribution statistics
# ============================================================

if df_cmi is not None and 'cmi' in df_cmi.columns:
    cmi_vals = df_cmi['cmi'].dropna()
    
    print('Hospital CMI Distribution')
    print('=' * 50)
    print(f'Hospitals with CMI data: {len(cmi_vals):,}')
    print(f'\nStatistics:')
    print(f'  Mean:   {cmi_vals.mean():.4f}')
    print(f'  Median: {cmi_vals.median():.4f}')
    print(f'  Std:    {cmi_vals.std():.4f}')
    print(f'  Min:    {cmi_vals.min():.4f}')
    print(f'  Max:    {cmi_vals.max():.4f}')
    print(f'  25th:   {cmi_vals.quantile(0.25):.4f}')
    print(f'  75th:   {cmi_vals.quantile(0.75):.4f}')
    
    # Distribution by quartile
    print(f'\nCMI Quartiles:')
    for q, label in [(0.25, 'Q1 (lowest)'), (0.50, 'Q2'), (0.75, 'Q3'), (1.0, 'Q4 (highest)')]:
        if q == 0.25:
            mask = cmi_vals <= cmi_vals.quantile(q)
        elif q == 1.0:
            mask = cmi_vals > cmi_vals.quantile(0.75)
        else:
            mask = (cmi_vals > cmi_vals.quantile(q - 0.25)) & (cmi_vals <= cmi_vals.quantile(q))
        print(f'  {label}: {mask.sum():,} hospitals (CMI range: {cmi_vals[mask].min():.3f} - {cmi_vals[mask].max():.3f})')
else:
    print('⚠️  CMI data not available yet. Run cells above to load the Impact/CMI file.')

Hospital CMI Distribution
Hospitals with CMI data: 3,151

Statistics:
  Mean:   1.7644
  Median: 1.6997
  Std:    0.4312
  Min:    0.6543
  Max:    4.8844
  25th:   1.5049
  75th:   1.9470

CMI Quartiles:
  Q1 (lowest): 788 hospitals (CMI range: 0.654 - 1.505)
  Q2: 788 hospitals (CMI range: 1.505 - 1.700)
  Q3: 787 hospitals (CMI range: 1.700 - 1.947)
  Q4 (highest): 788 hospitals (CMI range: 1.947 - 4.884)


## 8. Save Output

We save two files:
1. **Hospital CMI data** — one row per hospital with CMI, discharges, and related fields
2. **DRG severity reference** — the DRG family/tier/weight mapping for revenue impact modeling

In [10]:
# ============================================================
# Save hospital-level CMI data
# ============================================================

if df_cmi is not None:
    # Drop rows with no CCN
    df_cmi_clean = df_cmi.dropna(subset=['ccn']).copy()
    
    output_path = OUTPUT_DIR / 'hospital_drg_data.csv'
    df_cmi_clean.to_csv(output_path, index=False)
    
    print(f'Saved {len(df_cmi_clean):,} hospitals to:')
    print(f'  {output_path}')
    print(f'  Columns: {list(df_cmi_clean.columns)}')
else:
    print('⚠️  No CMI data to save. Download the Impact File first.')

# Also save the severity tier reference with payment gaps
severity_ref_path = OUTPUT_DIR / 'severity_tier_payments.csv'
severity_pivot.to_csv(severity_ref_path)
print(f'\nSaved severity tier reference to:')
print(f'  {severity_ref_path}')

print(f'\n✓ NB02 complete.')
print(f'  → hospital_drg_data.csv will be joined with NB01 hospital characteristics in NB07')
print(f'  → DRG severity reference will be used for revenue impact modeling in NB08')

Saved 3,151 hospitals to:
  /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb02_drg_cmi/hospital_drg_data.csv
  Columns: ['ccn', 'hospital_name', 'cmi', 'discharges', 'wage_index', 'beds', 'teaching_pct', 'dsh_pct']

Saved severity tier reference to:
  /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb02_drg_cmi/severity_tier_payments.csv

✓ NB02 complete.
  → hospital_drg_data.csv will be joined with NB01 hospital characteristics in NB07
  → DRG severity reference will be used for revenue impact modeling in NB08
